In [ ]:
# ============================================================
# Colab Setup & Environment Verification
# ============================================================
import os, sys

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print(f"Environment: {'Google Colab' if IN_COLAB else 'Local Environment'}")


# Signature Verification System — Interactive Pipeline

End-to-end **spatial + temporal** workflow for the Multimodal Signature Verification
System (EE 499).

Every heavy operation in this notebook is **imported from `scripts/`, not
re-implemented**: the training loop is `scripts.train.fit`, the metrics are
`scripts.evaluate.compute_eer` / `compute_far_frr`, and the ROC/DET/confusion
plots are `scripts.evaluate.plot_*`. Running this notebook and running
`python scripts/train.py` execute the same code, so the two cannot drift apart.

**Order of operations**

1. **Setup** — clone + install (Colab only), then imports and device selection
2. **Data** — discover datasets, preview genuine vs skilled-forgery pairs
3. **Split** — ISO/IEC 23053 writer-independent partition, then pairing
4. **Train** — spatial CNN, with live `tqdm` progress and a loss curve after
5. **Evaluate** — EER / FAR / FRR, confusion matrix, ROC and DET curves
6. **Temporal** — SVC2004 BiLSTM branch, same train/evaluate path
7. **Fuse** — late-fusion decision engine across both branches

> **Figures are not stored in this notebook.** Cell outputs are stripped so the file stays reviewable in a diff; every plot this pipeline generates is written to `results/` when the cells run, and the headline numbers are tabulated in `README.md`.


In [ ]:
import os
import sys
import random

import numpy as np
import cv2
import torch
import torch.optim as optim
import matplotlib.pyplot as plt

# Add project root directory to path
sys.path.insert(0, os.path.abspath('.'))

from config.settings import (
    SPATIAL_EMBEDDING_DIM, TEMPORAL_EMBEDDING_DIM,
    CONTRASTIVE_MARGIN, LEARNING_RATE, BATCH_SIZE,
    NUM_EPOCHS, WEIGHT_DECAY, MODEL_WEIGHTS_DIR,
    DECISION_THRESHOLD, SPATIAL_DECISION_THRESHOLD, TEMPORAL_DECISION_THRESHOLD,
    TARGET_FAR, TARGET_FRR,
    HARD_NEGATIVE_RATIO, DATA_DIR,
    CEDAR_DIR, BHSIG260_DIR, UTSIG_DIR,
    SVC2004_TASK2_DIR, SIGCOMP2011_DIR,
    SPATIAL_SPLIT_RATIOS, TEMPORAL_SPLIT_RATIOS,
    TEMPORAL_HIDDEN_SIZE, TEMPORAL_NUM_LAYERS, PRESSURE_DROPOUT_RATE,
    SPATIAL_TRUNK, SPATIAL_CONV_DROPOUT, SPATIAL_FC_HIDDEN, SPATIAL_GRAD_CHECKPOINT,
    ENROLLMENT_TRIALS, NUM_ENROLLMENT_REFERENCES,
)
from src.models.siamese import SpatialSiamese, TemporalSiamese, ContrastiveLoss, FusionEngine
from src.data.dataset import preprocess_image, writer_independent_split, generate_skilled_pairs
from src.data.parsers import SPATIAL_DATASETS, parse_dataset

from scripts.train import build_loader, build_writer_samples, fit
from scripts.evaluate import (
    score_pairs, compute_eer, compute_far_frr,
    plot_loss_curve, plot_roc, plot_det, plot_confusion_matrix,
    embed_writers, enrolled_trial_scores, enrolled_writer_count,
    enrolled_split_scores,
)
from scripts.benchmark import enrolled_scores

# Random seeds for reproducible evaluation
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# DataLoader worker processes — must be 0 in notebooks
NUM_WORKERS = 0

# Dynamic hardware device selection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware Device Selected: {device}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("VRAM memory cache cleared successfully.")


In [ ]:
# Load real datasets through the same function the CLI uses.
#
# `build_writer_samples` is scripts/train.py's loader. It calls parse_dataset per
# directory and merges with `merge_datasets`, which APPENDS per writer. A plain
# dict.update() would replace a writer's whole record when two datasets share an
# ID, silently dropping their samples — see "Silent data loss when merging
# datasets" in docs/SOLUTIONS.md. Today's IDs are namespaced (cedar_1,
# bhsig_Bengali_001, utsig_1) so nothing collides, but reimplementing the merge
# here would reintroduce the hazard the moment a dataset ships bare numeric IDs.
#
# parse_dataset also handles each dataset's real on-disk layout. An os.walk()
# that treats any image directory as one "writer" would collapse all 55 CEDAR
# signers into a single entry and destroy writer independence outright.
# Imported, not restated: the local copy here had already drifted from
# parsers.SPATIAL_DATASETS, which also lists sigcomp2011_offline. A
# notebook run silently trained on a smaller corpus than the CLI did.

spatial_dirs = [os.path.join(DATA_DIR, name) for name in SPATIAL_DATASETS]
spatial_dirs = [d for d in spatial_dirs if os.path.isdir(d)]

samples_by_writer = build_writer_samples(spatial_dirs) if spatial_dirs else {}
print(f"Total: {len(samples_by_writer)} writers")

# Fallback: generate a small synthetic dataset so the notebook runs without data
if not samples_by_writer:
    print("No datasets found under ./data. Generating synthetic signature pairs for demonstration...")
    demo_root = os.path.join(DATA_DIR, "demo_dataset")
    os.makedirs(demo_root, exist_ok=True)

    for writer in range(1, 5):
        genuine, forged = [], []
        for i in range(1, 6):
            img = np.ones((256, 256), dtype=np.uint8) * 255
            cv2.line(img, (30 + i * 5, 128), (220, 128 + writer * 4), 0, 3)
            path = f"{demo_root}/w{writer}_g{i}.png"
            cv2.imwrite(path, img)
            genuine.append(path)

            img = np.ones((256, 256), dtype=np.uint8) * 255
            cv2.line(img, (34 + i * 5, 132), (216, 124 + writer * 4), 0, 3)
            path = f"{demo_root}/w{writer}_f{i}.png"
            cv2.imwrite(path, img)
            forged.append(path)

        samples_by_writer[f"demo_{writer}"] = {"genuine": genuine, "forged": forged}
    print(f"Demo dataset created with {len(samples_by_writer)} writers.")

# Generate sample pairs for preview visualization (skilled-forgery negatives)
demo_pairs, demo_labels = generate_skilled_pairs(
    samples_by_writer, num_genuine_per_writer=2, num_forged_per_writer=2)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle("Dataset Sample Preview — Genuine vs Skilled-Forgery Pairs", fontsize=14, fontweight="bold")

preview = list(zip(demo_pairs, demo_labels))
random.Random(42).shuffle(preview)

for ax, ((p1, p2), label) in zip(axes.flat, preview[:8]):
    img1 = preprocess_image(p1)[0]
    img2 = preprocess_image(p2)[0]
    ax.imshow(np.hstack([img1, img2]), cmap="gray")
    status = "Genuine Pair (y=1)" if label == 1 else "Skilled Forgery (y=0)"
    ax.set_title(status, color="green" if label == 1 else "red", fontsize=10, fontweight="bold")
    ax.axis("off")

plt.tight_layout()
os.makedirs("./results", exist_ok=True)
plt.savefig("./results/dataset_samples.png", dpi=150)
plt.show()


### What each offline corpus contributes, and what is left out

The project uses **every genuine signature** a corpus ships and **only its skilled
forgeries**: signatures produced by a forger who had the genuine in front of them
and practised it. The corpora define which forgeries are skilled; the project does
not pick them. What is deliberately dropped is the easy material that would flatter
the error rate:

| corpus | writers | per writer | dropped |
|---|---|---|---|
| CEDAR | 55 | 24 genuine + 24 skilled forgeries | nothing |
| BHSig260 Bengali / Hindi | 100 / 160 | 24 genuine + 30 skilled forgeries | nothing |
| UTSig (Persian) | 115 | 27 genuine + 6 skilled forgeries | its 36 *simple* and 3 *opposite-hand* forgeries per writer (`parse_utsig` defaults to `forgery_types=("Skilled",)`) |
| SigComp2011 offline | 20 paired writers | scans of the pen signings | evaluation only after 2026-09-18 (Part 9 §27) |

Pairs are built **after** the writer split, per writer: genuine-genuine positives
and genuine-vs-skilled-forgery negatives, capped by the scarcer side so every
writer is class-balanced (UTSig is capped at 27 × 6 = 162 pairs, the others at
C(24, 2) = 276, which is why `PAIRS_*_PER_WRITER = 276` is a ceiling, not a
tunable). Half of the negatives are random impostors (another writer's genuine),
which is the supervision that teaches identity rather than "does this look
forged". The cell below shows one genuine and one skilled forgery of the same
writer from each corpus, exactly as the network sees them.

In [ ]:
# ============================================================
# One example per offline corpus: a genuine and a skilled forgery of the same writer
# ============================================================
# Writer IDs are namespaced by corpus (cedar_1, bhsig_Bengali_001, utsig_1, ...),
# so the first writer under each prefix is the example. preprocess_image is the
# same call the training loader makes: resized to SPATIAL_INPUT_SIZE and inverted
# so ink is bright on black, which is what the network sees.
corpus_prefixes = [("cedar_", "CEDAR"), ("bhsig_Bengali_", "BHSig260 Bengali"),
                   ("bhsig_Hindi_", "BHSig260 Hindi"), ("utsig_", "UTSig"),
                   ("sigcomp_", "SigComp2011 offline")]
rows = [(name, next((w for w in sorted(samples_by_writer) if w.startswith(p)), None))
        for p, name in corpus_prefixes]
rows = [(name, w) for name, w in rows if w and samples_by_writer[w]["genuine"] and samples_by_writer[w]["forged"]]
if rows:
    fig, axes = plt.subplots(len(rows), 2, figsize=(9, 2.3 * len(rows)))
    axes = np.atleast_2d(axes)
    for r, (name, w) in enumerate(rows):
        rec = samples_by_writer[w]
        for c, (kind, path) in enumerate([("genuine", rec["genuine"][0]), ("skilled forgery", rec["forged"][0])]):
            axes[r, c].imshow(preprocess_image(path)[0], cmap="gray", vmin=0, vmax=1)
            axes[r, c].set_title(f"{name} · {w} · {kind}  ({len(rec['genuine'])} G / {len(rec['forged'])} F on disk)", fontsize=8)
            axes[r, c].axis("off")
    plt.tight_layout()
    os.makedirs("./results", exist_ok=True)
    plt.savefig("./results/dataset_examples_spatial.png", dpi=150)
    plt.show()
else:
    print("No offline corpus found under data/; skipping the examples.")

In [ ]:
# Perform ISO/IEC 23053 writer-independent data partition split
writer_ids = sorted(samples_by_writer.keys())
train_ids, val_ids, test_ids = writer_independent_split(
    writer_ids, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=SEED)

print(f"Writer Split Ratios: {len(train_ids)} Train | {len(val_ids)} Val | {len(test_ids)} Test")

# build_loader() is scripts/train.py's loader factory. It generates pairs AFTER
# the writer split (so no signer straddles a partition boundary) and uses one
# seed for every split — the same seed the CLI uses, so weights trained here and
# scored by scripts/evaluate.py see byte-identical pairs.
train_loader, n_train = build_loader(samples_by_writer, train_ids, "spatial",
                                     BATCH_SIZE, NUM_WORKERS, True, SEED)
val_loader, n_val = build_loader(samples_by_writer, val_ids or train_ids, "spatial",
                                 BATCH_SIZE, NUM_WORKERS, False, SEED)
test_loader, n_test = build_loader(samples_by_writer, test_ids or train_ids, "spatial",
                                   BATCH_SIZE, NUM_WORKERS, False, SEED)

print(f"Generated Pairs: {n_train} Train | {n_val} Val | {n_test} Test")


In [ ]:
# ============================================================
# Spatial branch — train (or load) SpatialSiamese
# ============================================================
# The epoch loop, per-epoch checkpointing, best-validation weight saving and
# early stopping all live in scripts/train.py::fit. This cell only decides
# whether to train, and plots the history fit() hands back.
weights_path = os.path.join(MODEL_WEIGHTS_DIR, "spatial_cnn.pth")
checkpoint_path = os.path.join(MODEL_WEIGHTS_DIR, "checkpoint_spatial_cnn.pth")
FORCE_TRAIN = False  # Set to True to force re-training

# Loading and training build the model differently, on purpose.
#
# Loading goes through `from_state_dict`, which reads the trunk, embedding width
# and FC head back out of the saved parameter shapes. A weights file outlives
# the config that produced it: `spatial_cnn.pth` is the wide trunk, and building
# from config instead would either throw a shape mismatch or, worse, score an
# archived checkpoint under an architecture it was never trained with. Every
# other load path in the project (benchmark.py, evaluate.py, verification.py)
# already goes through this, per AGENTS.md.
#
# Training is the one case where config is authoritative, since there are no
# saved shapes to read yet.
if os.path.exists(weights_path) and not FORCE_TRAIN:
    print(f"Pre-compiled model weights found at: {weights_path}")
    model = SpatialSiamese.from_state_dict(
        torch.load(weights_path, map_location=device)).to(device)
    print(f"Weights loaded. Architecture recovered from the checkpoint: "
          f"trunk={model.trunk}, embedding={model.embedding_dim}-d. "
          f"Skipping training phase.")
else:
    model = SpatialSiamese(embedding_dim=SPATIAL_EMBEDDING_DIM,
                           trunk=SPATIAL_TRUNK,
                           conv_dropout=SPATIAL_CONV_DROPOUT,
                           fc_hidden=SPATIAL_FC_HIDDEN,
                           grad_checkpoint=SPATIAL_GRAD_CHECKPOINT).to(device)

loss_fn = ContrastiveLoss(margin=CONTRASTIVE_MARGIN, hard_negative_ratio=HARD_NEGATIVE_RATIO)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

history = None
if not (os.path.exists(weights_path) and not FORCE_TRAIN):
    print(f"Starting training phase for {NUM_EPOCHS} epochs "
          f"(trunk={SPATIAL_TRUNK}, grad checkpoint={SPATIAL_GRAD_CHECKPOINT})...")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    history = fit(model, train_loader, val_loader, loss_fn, optimizer, device,
                  NUM_EPOCHS, weights_path, checkpoint_path)

    print(f"Training complete. Best model weights saved to: {weights_path}")

# Loss curve — how training and validation loss changed over the epochs just run.
# Drawn after fit() returns so the tqdm bars above stay intact. plot_loss_curve
# is shared with the temporal branch and marks the best-validation epoch, which
# is the one whose weights were actually saved.
if history:
    fig, ax = plt.subplots(figsize=(8, 5))
    plot_loss_curve(history, "Spatial CNN — Training vs Validation Loss", ax=ax)
    fig.tight_layout()
    os.makedirs("./results", exist_ok=True)
    fig.savefig("./results/training_loss.png", dpi=150)
    plt.show()
else:
    print("Loaded pre-trained weights — no loss curve to draw. "
          "Set FORCE_TRAIN = True to retrain and see it.")


In [ ]:
# ============================================================
# Spatial branch — evaluation (enrolled protocol matching evaluate.py & settings.py)
# ============================================================
class DummyArgs:
    batch_size = BATCH_SIZE
    seed = SEED

genuine_scores, impostor_scores = enrolled_split_scores(
    model, "spatial", samples_by_writer, test_ids, DummyArgs(), device, desc="test")

eer, eer_threshold = compute_eer(genuine_scores, impostor_scores)
tau_val = SPATIAL_DECISION_THRESHOLD
far_val, frr_val = compute_far_frr(genuine_scores, impostor_scores, tau_val)
far_fixed, frr_fixed = compute_far_frr(genuine_scores, impostor_scores, DECISION_THRESHOLD)
accuracy = np.mean((genuine_scores >= tau_val).tolist() + (impostor_scores < tau_val).tolist())

print("=" * 66)
print("                       SPATIAL EVALUATION REPORT")
print("=" * 66)
print(f"  Model: trunk={SPATIAL_TRUNK}, {SPATIAL_EMBEDDING_DIM}-d embedding")
print(f"  ENROLLMENT PROTOCOL  (K={NUM_ENROLLMENT_REFERENCES} refs, {ENROLLMENT_TRIALS} trials, {len(test_ids)} test writers)")
print(f"    Equal Error Rate:    {eer * 100:.2f}%   (τ @ EER: {eer_threshold:.4f})")
print(f"    Accuracy:            {accuracy * 100:.2f}%   (at τ_val={tau_val:.4f})")
print("-" * 66)
print(f"  {'':<28}{'τ_val':>12}{'τ_fixed':>14}")
print(f"  {'threshold':<28}{tau_val:>12.4f}{DECISION_THRESHOLD:>14.4f}")
print(f"  {'False Acceptance (FAR) %':<28}{far_val * 100:>12.2f}{far_fixed * 100:>14.2f}   target <{TARGET_FAR * 100:.0f}%")
print(f"  {'False Rejection (FRR) %':<28}{frr_val * 100:>12.2f}{frr_fixed * 100:>14.2f}   target <{TARGET_FRR * 100:.0f}%")
print("=" * 66)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_confusion_matrix(genuine_scores, impostor_scores, tau_val,
                      f"Spatial Confusion (τ={tau_val:.3f})", ax=axes[0])
plot_roc(genuine_scores, impostor_scores, f"Spatial ROC (EER={eer * 100:.2f}%)", ax=axes[1])
plot_det(genuine_scores, impostor_scores, "Spatial DET", ax=axes[2])
plt.tight_layout()
os.makedirs("./results", exist_ok=True)
plt.savefig("./results/confusion_spatial.png", dpi=150)
plt.show()


In [ ]:
# Protocol demonstration complete. All metrics and plots are generated above.


## Temporal Branch (SVC2004) + Multimodal Fusion

The cells above cover the **spatial** branch (offline signature *images*). This
section adds the **temporal** branch — online, dynamic signatures from **SVC2004**,
where each sample is a pen trajectory of `(X, Y, Pressure)` points — and then
demonstrates the **late-fusion** decision engine.

1. **Temporal data** — parse SVC2004, writer-independent split, variable-length
   trajectory pairs (padded + packed via `collate_temporal_pairs`).
2. **Temporal training** — `TemporalSiamese` (BiLSTM) → `temporal_lstm.pth`.
3. **Temporal evaluation** — EER / FAR / FRR on the held-out temporal writers.
4. **Fusion demo** — `FusionEngine` combining spatial + temporal scores, with
   unimodal fallback and degraded-mode routing.

> **Note on fusion scope.** The offline image datasets and SVC2004 are *disjoint* —
> no single signing event exists as both a rendered image **and** a trajectory —
> so there is no paired multimodal test set to benchmark end-to-end. The fusion
> cell therefore demonstrates the real `FusionEngine` **decision logic** (α-weighting,
> routing, degraded flags) on each trained branch's actual score magnitudes.

In [ ]:
# ============================================================
# Temporal branch data setup & splits
# ============================================================
# Load both pen-tablet datasets (svc2004_task2 and sigcomp2011) to match scripts/evaluate.py
from config.settings import TEMPORAL_CHANNELS

temporal_dirs = [SVC2004_TASK2_DIR, SIGCOMP2011_DIR]
discovered_pen_dirs = [d for d in temporal_dirs if os.path.exists(d)]

if not discovered_pen_dirs:
    print('WARNING: No pen-tablet dataset directories discovered in data/.')
    print('         Skipping temporal training & evaluation phases.')
    TEMPORAL_AVAILABLE = False
else:
    TEMPORAL_AVAILABLE = True
    print('Discovering temporal writers:')
    # Match scripts/evaluate.py: load whatever width TEMPORAL_CHANNELS selects.
    # Omitting this returned 3-channel `xyp` trajectories while the shipped
    # temporal_lstm.pth is 4-channel `dt`, so the notebook silently evaluated a
    # contract the project stopped shipping on 2026-08-19.
    temporal_writer_samples = build_writer_samples(
        discovered_pen_dirs,
        None if TEMPORAL_CHANNELS == "xyp" else TEMPORAL_CHANNELS)
    t_writer_ids = sorted(temporal_writer_samples.keys())

    t_train_ids, t_val_ids, t_test_ids = writer_independent_split(
        t_writer_ids, *TEMPORAL_SPLIT_RATIOS, seed=SEED)
    print(f'  Pen writers discovered: {len(t_writer_ids)}')
    print(f'  Temporal split: {len(t_train_ids)} train | {len(t_val_ids)} val | {len(t_test_ids)} test writers')


### What each pen corpus contributes, and what the model actually receives

| corpus | writers | per writer | used as |
|---|---|---|---|
| SVC2004 Task 2 | 40 | 20 genuine + 20 skilled forgeries, with real pressure | training and held-out evaluation |
| SigComp2011 online | 64 Dutch + 20 Chinese | genuine + skilled forgeries, 200 Hz decimated to 100 Hz | training and held-out evaluation |
| MOBISIG (touchscreen), DeepSignDB, SigComp2009 / NFI | 83 / 498 / 14 | | evaluation only, never trained on |

A pen signature is not an image. It enters the model as a `[L, C]` float array:
one row per sample at 100 Hz, `C = TEMPORAL_INPUT_FEATURES` columns, which for the
shipped `dt` contract are x, y, pressure and the clipped inter-sample gap. The
DataLoader pads a batch of pairs to its longest member and passes the true
lengths, and inside `TemporalSiamese` each row gets its delta to the previous
row appended (`_add_dynamics`, so `[L, 2C]`): velocity and pressure change are
what the LSTM sees. A skilled forgery is copied slowly, so it is typically two to
four times longer in samples and flatter in pressure than the genuine, which is
information a scan cannot carry. The cell below prints the first rows of the
array at each of those three stages for one genuine and one skilled forgery per
corpus.

In [ ]:
# ============================================================
# One example per pen corpus, as the tensors the model actually receives
# ============================================================
# No pictures: a trajectory is a [L, C] float array. C = TEMPORAL_INPUT_FEATURES
# (x, y, pressure, dt under the shipped "dt" contract), L = number of samples.
# Three views, in the order the data moves:
#   1. the parsed sample, first rows of the [L, C] array (what the parser emits)
#   2. the collated batch: pairs padded to the longest member, with true lengths,
#      which is what the DataLoader hands to the model
#   3. TemporalSiamese._add_dynamics: per-frame deltas appended, [L, 2C], which is
#      the sequence the LSTM consumes
if TEMPORAL_AVAILABLE:
    from src.data.dataset import collate_temporal_pairs
    np.set_printoptions(precision=4, suppress=True, linewidth=120)
    channel_names = {3: ["x", "y", "p"], 4: ["x", "y", "p", "dt"], 5: ["x", "y", "p", "azimuth", "altitude"]}
    pen_prefixes = [("svc_", "SVC2004 Task 2"), ("sigcomp_Dutch_", "SigComp2011 Dutch"),
                    ("sigcomp_Chinese_", "SigComp2011 Chinese")]
    for prefix, name in pen_prefixes:
        w = next((w for w in t_writer_ids if w.startswith(prefix)), None)
        if w is None:
            continue
        rec = temporal_writer_samples[w]
        for kind, seq in (("genuine", rec["genuine"][0]), ("skilled forgery", rec["forged"][0])):
            seq = np.asarray(seq, dtype=np.float32)
            print(f"{name} · {w} · {kind}: shape {seq.shape}  (L={len(seq)} samples, C={seq.shape[1]} channels "
                  f"{channel_names.get(seq.shape[1], '?')}), pen-up samples: {int((seq[:, 2] == 0).sum())}")
            print(seq[:6]); print("   ...")
        print()

    # 2. what one batch looks like after collate_temporal_pairs
    w = next((w for w in t_writer_ids if w.startswith("svc_")), t_writer_ids[0])
    g = [np.asarray(s, dtype=np.float32) for s in temporal_writer_samples[w]["genuine"][:2]]
    f = np.asarray(temporal_writer_samples[w]["forged"][0], dtype=np.float32)
    batch = [(torch.tensor(g[0]), torch.tensor(g[1]), torch.tensor(1.0)),   # genuine pair, label 1
             (torch.tensor(g[0]), torch.tensor(f), torch.tensor(0.0))]      # genuine vs skilled forgery, label 0
    x1, len1, x2, len2, labels = collate_temporal_pairs(batch)
    print(f"collated batch: x1 {tuple(x1.shape)}  x2 {tuple(x2.shape)}  lengths1 {len1.tolist()}  lengths2 {len2.tolist()}  labels {labels.tolist()}")
    print("  (padded to the longest member; pack_padded_sequence uses the lengths so pad rows never reach the LSTM)")

    # 3. what the LSTM consumes: the sample plus its per-frame deltas
    with torch.no_grad():
        inside = TemporalSiamese._add_dynamics(torch.tensor(g[0])[None])[0]
    print(f"\ninside the model, after _add_dynamics: {tuple(inside.shape)} = [L, 2C]: "
          f"{channel_names.get(g[0].shape[1], ['c'] * g[0].shape[1])} + their deltas (velocity and pressure change)")
    print(inside[:6].numpy())
else:
    print("No pen corpus found under data/; skipping the examples.")

In [ ]:
# ============================================================
# Temporal branch — train (or load) TemporalSiamese (BiLSTM)
# ============================================================
temporal_model = None
t_history = None
temporal_weights_name = "temporal_lstm.pth"

if not TEMPORAL_AVAILABLE:
    print("Temporal data unavailable — skipping temporal training.")
else:
    temporal_weights_path = os.path.join(MODEL_WEIGHTS_DIR, temporal_weights_name)
    temporal_checkpoint_path = os.path.join(
        MODEL_WEIGHTS_DIR, f"checkpoint_{os.path.splitext(temporal_weights_name)[0]}.pth")
    FORCE_TRAIN_TEMPORAL = False  # Set to True to force re-training

    if os.path.exists(temporal_weights_path) and not FORCE_TRAIN_TEMPORAL:
        print(f"Pre-compiled temporal weights found at: {temporal_weights_path}")
        temporal_model = TemporalSiamese.from_state_dict(
            torch.load(temporal_weights_path, map_location=device,
                       weights_only=True)).to(device)
        print("Weights loaded successfully. Skipping temporal training phase.")
    else:
        # This branch used to call
        #   fit(temporal_model, t_train_loader, t_val_loader, temporal_loss_fn,
        #       temporal_optimizer, ...)
        # and NONE of those four names is defined anywhere in this notebook —
        # `temporal_model` is still None on this path. It raised NameError, not a
        # usable message, and only stayed hidden because temporal_lstm.pth exists
        # on a developer box. Training belongs to scripts/train.py, which the
        # notebook is meant to reuse rather than re-implement.
        raise FileNotFoundError(
            f"No temporal weights at {temporal_weights_path}.\n"
            f"This notebook demonstrates the pipeline; it does not train the "
            f"temporal branch. Either download the release weights (see README) "
            f"or run:\n"
            f"  python scripts/train.py --branch temporal "
            f"--temporal-channels {TEMPORAL_CHANNELS} "
            f"--data-dir {' '.join(discovered_pen_dirs)}")

if t_history:
    fig, ax = plt.subplots(figsize=(8, 5))
    plot_loss_curve(t_history, "Temporal BiLSTM — Training vs Validation Loss", ax=ax)
    fig.tight_layout()
    os.makedirs("./results", exist_ok=True)
    fig.savefig("./results/temporal_training_loss.png", dpi=150)
    plt.show()


In [ ]:
# ============================================================
# Temporal branch — evaluation (enrolled protocol matching evaluate.py & settings.py)
# ============================================================
if not TEMPORAL_AVAILABLE:
    print("Temporal data unavailable — skipping temporal evaluation.")
else:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    class DummyArgs:
        batch_size = BATCH_SIZE
        seed = SEED

    t_genuine_scores, t_impostor_scores = enrolled_split_scores(
        temporal_model, "temporal", temporal_writer_samples, t_test_ids, DummyArgs(), device, desc="test")

    t_eer, t_eer_threshold = compute_eer(t_genuine_scores, t_impostor_scores)
    t_tau_val = TEMPORAL_DECISION_THRESHOLD
    t_far_val, t_frr_val = compute_far_frr(t_genuine_scores, t_impostor_scores, t_tau_val)
    t_far_fixed, t_frr_fixed = compute_far_frr(t_genuine_scores, t_impostor_scores, DECISION_THRESHOLD)
    t_accuracy = np.mean((t_genuine_scores >= t_tau_val).tolist() + (t_impostor_scores < t_tau_val).tolist())

    print("=" * 66)
    print("                 TEMPORAL EVALUATION REPORT")
    print("=" * 66)
    print(f"  Test writers:          {len(t_test_ids)}  ({len(t_genuine_scores)} genuine / {len(t_impostor_scores)} impostor probes)")
    print(f"  Test Accuracy:         {t_accuracy * 100:.2f}%   (at τ_val={t_tau_val:.4f})")
    print(f"  Equal Error Rate:      {t_eer * 100:.2f}%   (τ @ EER: {t_eer_threshold:.4f})")
    print("-" * 66)
    print(f"  {'':<28}{'τ_val':>12}{'τ_fixed':>14}")
    print(f"  {'threshold':<28}{t_tau_val:>12.4f}{DECISION_THRESHOLD:>14.4f}")
    print(f"  {'False Acceptance (FAR) %':<28}{t_far_val * 100:>12.2f}{t_far_fixed * 100:>14.2f}   target <{TARGET_FAR * 100:.0f}%")
    print(f"  {'False Rejection (FRR) %':<28}{t_frr_val * 100:>12.2f}{t_frr_fixed * 100:>14.2f}   target <{TARGET_FRR * 100:.0f}%")
    print("=" * 66)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    plot_confusion_matrix(t_genuine_scores, t_impostor_scores, t_tau_val,
                          f"Temporal Confusion (τ={t_tau_val:.3f})", ax=axes[0])
    plot_roc(t_genuine_scores, t_impostor_scores, f"Temporal ROC (EER={t_eer * 100:.2f}%)", ax=axes[1])
    plot_det(t_genuine_scores, t_impostor_scores, "Temporal DET", ax=axes[2])
    plt.tight_layout()
    os.makedirs("./results", exist_ok=True)
    plt.savefig("./results/confusion_temporal.png", dpi=150)
    plt.show()



### SignatureService — the shipped serving facade, all three modes

---
 Every cell above scores a branch in isolation, which is how the model is
 *measured*. This cell exercises the class the GUI and database layers
 actually call: `src.verification.SignatureService`.


In [ ]:

from src.verification import SignatureService
from src.data.renderer import render_trajectory_to_image

_shipped = ("spatial_cnn.pth", "temporal_lstm.pth")
_weights_present = all(os.path.exists(os.path.join(MODEL_WEIGHTS_DIR, w)) for w in _shipped)

if not (TEMPORAL_AVAILABLE and _weights_present):
    print("SignatureService demo needs the shipped weights and pen data — skipping.")
    print(f"  weights present: {_weights_present}   pen data: {TEMPORAL_AVAILABLE}")
else:
    service = SignatureService(device=str(device))

    # A held-out writer with enough genuine samples to enrol K references and
    # still hold a probe back. References and probes stay disjoint, the same
    # rule the enrolled protocol in scripts/evaluate.py follows.
    subject = next(
        (w for w in t_test_ids
         if len(temporal_writer_samples[w]["genuine"]) > NUM_ENROLLMENT_REFERENCES
         and temporal_writer_samples[w]["forged"]),
        None)

    if subject is None:
        print("No held-out writer has enough genuine samples to enrol — skipping.")
    else:
        genuine = temporal_writer_samples[subject]["genuine"]
        forged = temporal_writer_samples[subject]["forged"]
        refs = genuine[:NUM_ENROLLMENT_REFERENCES]
        probe_gen = genuine[NUM_ENROLLMENT_REFERENCES]
        probe_forg = forged[0]

        # One call, both template sets, from the same signing event.
        templates = service.enroll_from_tablet(refs)
        ref_t = templates["temporal_embeddings"]
        ref_s = templates["spatial_embeddings"]
        print(f"Enrolled writer {subject}: {len(ref_t)} temporal + {len(ref_s)} spatial "
              f"templates from {len(refs)} pen signings")
        print(f"Probes held out: 1 genuine, 1 skilled forgery (never enrolled)")
        print("=" * 78)

        def show(label, **kwargs):
            """Print one verification, including why it degraded or was refused."""
            try:
                r = service.verify(**kwargs)
            except ValueError as exc:
                print(f"  {label:<20} refused — {exc}")
                return
            flag = "  DEGRADED" if r.get("degraded") else ""
            print(f"  {label:<20} {r['decision']:<8} confidence={r['confidence']:6.2f}  "
                  f"routing={r['routing']}{flag}")
            for reason in r.get("reasons", []):
                print(f"       - {reason}")

        print("\nmode='unimodal'  — temporal branch only")
        show("genuine probe", mode="unimodal", probe_trajectory=probe_gen,
             reference_temporal=ref_t)
        show("skilled forgery", mode="unimodal", probe_trajectory=probe_forg,
             reference_temporal=ref_t)

        print("\nmode='temporal-multi'  — one pen event auto-rendered into both branches")
        show("genuine probe", mode="temporal-multi", probe_trajectory=probe_gen,
             reference_temporal=ref_t, reference_spatial=ref_s)
        show("skilled forgery", mode="temporal-multi", probe_trajectory=probe_forg,
             reference_temporal=ref_t, reference_spatial=ref_s)

        probe_img_gen = render_trajectory_to_image(probe_gen)
        probe_img_forg = render_trajectory_to_image(probe_forg)

        print("\nmode='multimodal'  — caller supplies the image explicitly")
        show("genuine probe", mode="multimodal", probe_trajectory=probe_gen,
             probe_image=probe_img_gen, reference_temporal=ref_t, reference_spatial=ref_s)
        show("skilled forgery", mode="multimodal", probe_trajectory=probe_forg,
             probe_image=probe_img_forg, reference_temporal=ref_t, reference_spatial=ref_s)

        print("\nmode='unimodal'  — spatial branch only, scoring the rendered probe")
        show("genuine probe", mode="unimodal", probe_image=probe_img_gen,
             reference_spatial=ref_s)
        show("skilled forgery", mode="unimodal", probe_image=probe_img_forg,
             reference_spatial=ref_s)

        print("=" * 78)
        print("Enrolment gates every sample it is handed, exactly as verify() gates every")
        print("probe: a scan or trajectory that fails its input gate raises here rather")
        print("than becoming a stored template no later comparison could be trusted against.")
